# Module 12: Deep Active Inference

---

In Module 11 we learned tabular A and B matrices from data. But tabular models
have a fundamental scaling problem: an A matrix for 100x100 grayscale images
would have $10{,}000 \times N_{states}$ entries -- and for color images or
continuous observations, it's worse.

**Deep Active Inference** replaces explicit tables with **neural networks**:

- **Encoder** $P(s|o)$: maps raw observations to state belief logits
- **Transition** $P(s'|s,a)$: predicts next-state distributions from beliefs + actions

This module covers:

1. The scaling problem with tabular models
2. Neural network generative models: encoder + transition
3. Amortized inference: encoder directly maps observations to beliefs
4. The **encoder collapse** problem: when the recognition model ignores its input
5. Encoder vs decoder architectures

### References

- Fountas et al. (2020). Deep Active Inference Agents Using Monte-Carlo Methods. NeurIPS.
- Millidge, Tschantz & Buckley (2020). Whence the Expected Free Energy?
- Tschantz, Millidge et al. (2020). Reinforcement Learning through Active Inference.

In [ ]:
# ── Imports ─────────────────────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt

import jax
import jax.numpy as jnp

from alf.deep_aif import (
    DeepGenerativeModel,
    learn_deep_model, DeepLearningResult,
    init_encoder, init_transition,
    encode, predict_transition,
    deep_analytic_nll,
    extract_A_matrix,
)
from alf.learning import (
    learn_model, generate_data, params_to_A,
)

import sys; sys.path.insert(0, '..')
from utils.plotting import (
    plot_matrix_heatmap, plot_learning_curve, COLORS,
)

np.set_printoptions(precision=3, suppress=True)
print(f"JAX version: {jax.__version__}")
print("All imports successful.")

## 1. The Scaling Problem

Let's start by making the scaling problem concrete. How large would the A matrix
be for different observation spaces?

In [ ]:
# ── Show the scaling problem ─────────────────────────────────────────────

scenarios = [
    ("T-maze (5 obs, 8 states)", 5, 8),
    ("Grid world (100 obs, 25 states)", 100, 25),
    ("Small image (32x32 grayscale)", 32*32, 16),
    ("Medium image (64x64 grayscale)", 64*64, 32),
    ("Large image (100x100 RGB)", 100*100*3, 64),
]

print(f"{'Scenario':40s} {'obs_dim':>10s} {'states':>8s} {'A entries':>12s} {'Memory (MB)':>12s}")
print("-" * 84)
for name, obs_dim, num_states in scenarios:
    entries = obs_dim * num_states
    memory_mb = entries * 8 / 1e6  # float64
    print(f"{name:40s} {obs_dim:10,d} {num_states:8d} {entries:12,d} {memory_mb:12.2f}")

print("\nFor images, explicit A matrices are impractical.")
print("Solution: replace A with a neural network encoder.")

## 2. The Deep Generative Model

The `DeepGenerativeModel` replaces explicit A and B matrices with MLPs:

**Encoder** (replaces A): `obs_vector -> MLP -> state_logits -> softmax -> P(s|o)`

**Transition** (replaces B): `(state_beliefs, action_onehot) -> MLP -> next_logits -> softmax -> P(s'|s,a)`

Everything is implemented in pure JAX -- no Flax, Haiku, or Equinox required.
Network parameters are stored as lists of (weight, bias) tuples for
transparent pytree handling by `jax.grad` and `jax.jit`.

In [ ]:
# ── Initialize a DeepGenerativeModel ─────────────────────────────────────

obs_dim = 5      # e.g., 5-dimensional observation vector
num_states = 4
num_actions = 3

deep_gm = DeepGenerativeModel(
    obs_dim=obs_dim,
    num_states=num_states,
    num_actions=num_actions,
    encoder_hidden=[32, 32],
    transition_hidden=[32, 32],
    seed=42,
)

print(f"Deep Generative Model:")
print(f"  obs_dim:     {deep_gm.obs_dim}")
print(f"  num_states:  {deep_gm.num_states}")
print(f"  num_actions: {deep_gm.num_actions}")

# Count parameters
enc_params = sum(w.size + b.size for w, b in deep_gm.encoder_params)
trans_params = sum(w.size + b.size for w, b in deep_gm.transition_params)
print(f"\nEncoder parameters:    {enc_params:,d}")
print(f"Transition parameters: {trans_params:,d}")
print(f"Total parameters:      {enc_params + trans_params:,d}")

# Compare with explicit A matrix
a_entries = obs_dim * num_states
b_entries = num_states * num_states * num_actions
print(f"\nExplicit A+B entries:  {a_entries + b_entries:,d}")
print(f"\nFor small problems, networks have MORE parameters (more expressive).")
print(f"For large obs_dim, networks scale MUCH better.")

In [ ]:
# ── Test the encoder and transition networks ─────────────────────────────

# Random observation vector
obs_test = jnp.array([1.0, 0.0, 0.0, 0.0, 0.0])  # one-hot for obs=0

# Encoder: map observation to state beliefs
state_logits = encode(deep_gm.encoder_params, obs_test)
state_beliefs = jax.nn.softmax(state_logits)

print("Encoder output:")
print(f"  Raw logits:  {state_logits.round(3)}")
print(f"  P(s|o):      {state_beliefs.round(3)}")
print(f"  Sum:         {float(state_beliefs.sum()):.6f}")

# Transition: predict next state
action_onehot = jax.nn.one_hot(0, num_actions)
next_logits = predict_transition(deep_gm.transition_params, state_beliefs, action_onehot)
next_beliefs = jax.nn.softmax(next_logits)

print(f"\nTransition output (action=0):")
print(f"  Raw logits:  {next_logits.round(3)}")
print(f"  P(s'|s,a):   {next_beliefs.round(3)}")

## 3. Training the Deep Model

We'll generate synthetic data from a known tabular model (as in Module 11)
and train the deep model to learn the same structure. This lets us compare
the deep model's learned representations against the known ground truth.

In [ ]:
# ── Generate synthetic data from a known tabular model ───────────────────

num_states_true = 3
obs_dim_true = 4
num_actions_true = 2

# True A matrix
A_true = np.array([
    [0.8, 0.1, 0.05],
    [0.1, 0.7, 0.1],
    [0.05, 0.1, 0.75],
    [0.05, 0.1, 0.1],
])

# True B matrix
B_true = np.zeros((num_states_true, num_states_true, num_actions_true))
B_true[:, :, 0] = np.array([[0.1, 0.1, 0.8], [0.8, 0.1, 0.1], [0.1, 0.8, 0.1]])
B_true[:, :, 1] = np.array([[0.8, 0.1, 0.1], [0.1, 0.8, 0.1], [0.1, 0.1, 0.8]])

# Prior
D_true = np.array([0.8, 0.1, 0.1])

# Generate data
T_data = 300
rng = np.random.RandomState(42)
actions_data = rng.randint(0, num_actions_true, size=T_data)

observations_idx, true_states = generate_data(
    A=A_true, B=B_true, D=D_true, actions=actions_data, seed=42,
)

# Convert observations to one-hot vectors for the deep model
observations_onehot = np.eye(obs_dim_true)[observations_idx]

# Pad actions
actions_padded = np.concatenate([actions_data, [0]])

print(f"Generated {len(observations_idx)} observations")
print(f"Observation one-hot shape: {observations_onehot.shape}")
print(f"Actions shape: {actions_padded.shape}")
print(f"\nFirst 5 one-hot observations:")
print(observations_onehot[:5])

In [ ]:
# ── Train the deep model ─────────────────────────────────────────────────

deep_result = learn_deep_model(
    observations_raw=observations_onehot,
    actions=actions_padded,
    obs_dim=obs_dim_true,
    num_states=num_states_true,
    num_actions=num_actions_true,
    D=D_true,
    encoder_hidden=[32, 32],
    transition_hidden=[32, 32],
    num_epochs=500,
    lr=0.001,
    seed=42,
    verbose=True,
)

print(f"\nFinal deep NLL: {deep_result.loss_history[-1]:.4f}")

In [ ]:
# ── Compare deep vs tabular learning curves ──────────────────────────────

# Also train a tabular model for comparison
tabular_result = learn_model(
    observations=observations_idx,
    actions=actions_padded,
    num_obs=obs_dim_true,
    num_states=num_states_true,
    num_actions=num_actions_true,
    D=D_true,
    num_epochs=500,
    lr=0.01,
    verbose=False,
)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(deep_result.loss_history, label='Deep model (neural net)',
        color=COLORS['aif'], linewidth=2)
ax.plot(tabular_result.loss_history, label='Tabular model (explicit A/B)',
        color=COLORS['rl'], linewidth=2)
ax.set_xlabel('Epoch')
ax.set_ylabel('Negative Log-Likelihood')
ax.set_title('Learning Curves: Deep vs Tabular')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Final tabular NLL: {tabular_result.loss_history[-1]:.4f}")
print(f"Final deep NLL:    {deep_result.loss_history[-1]:.4f}")
print(f"\nFor small discrete problems, tabular learning is more sample-efficient.")
print(f"The deep model's advantage emerges with high-dimensional observations.")

## 4. Inspecting Learned Representations

What did the encoder learn? For discrete observations encoded as one-hot vectors,
we can extract an approximate A matrix by passing each one-hot vector through
the encoder. This gives us a "recognition model" $P(s|o)$ -- the transpose
relationship to the generative model's $P(o|s)$.

In [ ]:
# ── Extract approximate A matrix from the encoder ────────────────────────

A_deep = extract_A_matrix(deep_result.encoder_params, obs_dim_true)

print("Extracted A matrix (recognition model P(s|o)):")
print(np.array(A_deep).round(3))
print(f"\nRow sums (each should = 1): {np.array(A_deep).sum(axis=1).round(3)}")
print("\nNote: This is P(s|o), not P(o|s). Each ROW sums to 1.")
print("Under uniform P(s), P(s|o) is proportional to P(o|s).")

In [ ]:
# ── Visualize encoder representations ────────────────────────────────────

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# True A matrix (generative model: P(o|s))
plot_matrix_heatmap(
    A_true,
    title="True A: P(o|s)",
    row_labels=[f"obs {i}" for i in range(obs_dim_true)],
    col_labels=[f"state {i}" for i in range(num_states_true)],
    cmap="YlOrRd",
    ax=axes[0],
)

# Tabular learned A (generative model: P(o|s))
plot_matrix_heatmap(
    tabular_result.learned_A[0],
    title="Tabular Learned A: P(o|s)",
    row_labels=[f"obs {i}" for i in range(obs_dim_true)],
    col_labels=[f"state {i}" for i in range(num_states_true)],
    cmap="YlOrRd",
    ax=axes[1],
)

# Deep encoder (recognition model: P(s|o))
plot_matrix_heatmap(
    np.array(A_deep),
    title="Deep Encoder: P(s|o)",
    row_labels=[f"obs {i}" for i in range(obs_dim_true)],
    col_labels=[f"state {i}" for i in range(num_states_true)],
    cmap="YlOrRd",
    ax=axes[2],
)

plt.suptitle("Comparing Representations: True vs Tabular vs Deep", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("The deep encoder learns P(s|o) -- the recognition direction.")
print("The tabular A is P(o|s) -- the generative direction.")
print("They encode the same information but in transposed form.")

## 5. Encoder Collapse

A critical failure mode in deep Active Inference: the **encoder collapse** problem.
This occurs when the encoder network ignores its input and maps all observations
to the same state belief. The transition network then does all the work, and the
encoder becomes degenerate.

This can happen when:
- The learning rate is too high
- The transition network is too powerful relative to the encoder
- There isn't enough diversity in the observation data

In [ ]:
# ── Demonstrate encoder collapse ─────────────────────────────────────────
#
# We intentionally create conditions that promote collapse:
# - Very high learning rate
# - Small encoder, large transition network

collapsed_result = learn_deep_model(
    observations_raw=observations_onehot,
    actions=actions_padded,
    obs_dim=obs_dim_true,
    num_states=num_states_true,
    num_actions=num_actions_true,
    D=D_true,
    encoder_hidden=[8],          # Very small encoder
    transition_hidden=[64, 64],  # Large transition network
    num_epochs=300,
    lr=0.01,                     # High learning rate
    seed=99,
    verbose=False,
)

# Check if encoder has collapsed
A_collapsed = extract_A_matrix(collapsed_result.encoder_params, obs_dim_true)

print("Encoder output for each observation:")
for o in range(obs_dim_true):
    obs_vec = jnp.eye(obs_dim_true)[o]
    beliefs = jax.nn.softmax(encode(collapsed_result.encoder_params, obs_vec))
    print(f"  obs {o}: P(s|o) = {np.array(beliefs).round(3)}")

# Measure collapse: if all outputs are similar, the encoder has collapsed
A_np = np.array(A_collapsed)
row_variance = np.var(A_np, axis=0).mean()
collapse_score = 1.0 - np.std(A_np, axis=0).mean()  # 1 = collapsed, 0 = diverse

print(f"\nCollapse diagnostics:")
print(f"  Mean row variance:  {row_variance:.6f}")
print(f"  Collapse score:     {collapse_score:.3f} (1.0 = fully collapsed)")

if row_variance < 0.01:
    print("\n  ENCODER COLLAPSE DETECTED: all observations map to similar beliefs.")
    print("  The encoder has learned to ignore its input.")
else:
    print("\n  No collapse detected (try different random seed or more extreme params).")
    print("  Collapse is stochastic -- it depends on initialization and optimization path.")

In [ ]:
# ── Visualize collapsed vs healthy encoder ───────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

plot_matrix_heatmap(
    np.array(A_deep),
    title="Healthy Encoder: P(s|o)",
    row_labels=[f"obs {i}" for i in range(obs_dim_true)],
    col_labels=[f"state {i}" for i in range(num_states_true)],
    cmap="YlOrRd",
    ax=axes[0],
)

plot_matrix_heatmap(
    np.array(A_collapsed),
    title="Collapsed/Degraded Encoder: P(s|o)",
    row_labels=[f"obs {i}" for i in range(obs_dim_true)],
    col_labels=[f"state {i}" for i in range(num_states_true)],
    cmap="YlOrRd",
    ax=axes[1],
)

plt.suptitle("Encoder Collapse: Healthy vs Degraded", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("In a healthy encoder, different observations map to different state beliefs.")
print("In a collapsed encoder, all observations produce the same (or very similar) beliefs.")

## 6. Encoder vs Decoder Architectures

There's a subtle but important design choice in deep Active Inference:

- **Encoder** $P(s|o)$: Recognition model. Maps observations to states.
  This is what our current implementation uses.
  
- **Decoder** $P(o|s)$: Generative model. Predicts observations from states.
  Used in VAE-style approaches.

The decoder architecture avoids collapse more naturally because:
1. The loss directly penalizes incorrect observation predictions
2. States that produce identical observations are immediately penalized
3. The decoder creates a "pull" on states to be meaningfully different

The encoder architecture is more prone to collapse because:
1. The loss is NLL of the full sequence (indirect supervision)
2. A collapsed encoder can still produce reasonable NLL if the transition network compensates
3. There's no direct per-observation reconstruction loss

In [ ]:
# ── Conceptual comparison: encoder vs decoder ────────────────────────────
#
# Our current DeepGenerativeModel uses an encoder P(s|o).
# Here we show what a decoder P(o|s) would look like.

print("=== Architecture Comparison ===")
print()
print("ENCODER ARCHITECTURE (current):")
print("  Input:  observation vector o")
print("  Output: state belief logits")
print("  Usage:  P(s|o) = softmax(encoder(o))")
print("  Risk:   Can ignore input (collapse)")
print()
print("DECODER ARCHITECTURE (VAE-style):")
print("  Input:  state belief vector s")
print("  Output: predicted observation logits")
print("  Usage:  P(o|s) = softmax(decoder(s))")
print("  Benefit: Reconstruction loss prevents collapse")
print()
print("In practice, BOTH can work well with careful regularization.")
print("The encoder approach is simpler (amortized inference).")
print("The decoder approach is more principled (direct generative model).")

## 7. Exercise: Effect of Hidden Dimensions

How does the network architecture affect learning? Try different hidden layer
configurations and observe the effect on:
1. Final NLL
2. Learning speed (convergence rate)
3. Encoder quality (collapse risk)

In [ ]:
# ── Exercise: vary hidden dimensions ─────────────────────────────────────

architectures = {
    "Tiny [8]": ([8], [8]),
    "Small [16,16]": ([16, 16], [16, 16]),
    "Medium [32,32]": ([32, 32], [32, 32]),
    "Large [64,64]": ([64, 64], [64, 64]),
    "Deep [16,16,16]": ([16, 16, 16], [16, 16, 16]),
}

arch_results = {}

for name, (enc_h, trans_h) in architectures.items():
    r = learn_deep_model(
        observations_raw=observations_onehot,
        actions=actions_padded,
        obs_dim=obs_dim_true,
        num_states=num_states_true,
        num_actions=num_actions_true,
        D=D_true,
        encoder_hidden=enc_h,
        transition_hidden=trans_h,
        num_epochs=400,
        lr=0.001,
        seed=42,
        verbose=False,
    )
    arch_results[name] = r
    
    # Count parameters
    n_params = sum(w.size + b.size for w, b in r.encoder_params)
    n_params += sum(w.size + b.size for w, b in r.transition_params)
    
    print(f"{name:25s}: final NLL = {r.loss_history[-1]:.4f}, params = {n_params:,d}")

In [ ]:
# ── Plot learning curves for all architectures ───────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Full learning curves
colors_list = [COLORS['rl'], COLORS['aif'], COLORS['reward'], COLORS['epistemic'], COLORS['highlight']]
for (name, r), color in zip(arch_results.items(), colors_list):
    axes[0].plot(r.loss_history, label=name, linewidth=1.5, color=color)

axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('NLL')
axes[0].set_title('Learning Curves by Architecture')
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

# Final NLL comparison
names = list(arch_results.keys())
final_nlls = [arch_results[n].loss_history[-1] for n in names]
axes[1].barh(range(len(names)), final_nlls, color=colors_list[:len(names)], alpha=0.8)
axes[1].set_yticks(range(len(names)))
axes[1].set_yticklabels(names, fontsize=9)
axes[1].set_xlabel('Final NLL')
axes[1].set_title('Final NLL by Architecture')

plt.tight_layout()
plt.show()

print("Observations:")
print("- Larger networks fit the data better but take longer to converge.")
print("- Very small networks may underfit.")
print("- The relationship is not always monotonic due to optimization landscape effects.")

In [ ]:
# ── Inspect encoder quality across architectures ─────────────────────────

fig, axes = plt.subplots(1, len(arch_results), figsize=(5 * len(arch_results), 4))

for idx, (name, r) in enumerate(arch_results.items()):
    A_enc = extract_A_matrix(r.encoder_params, obs_dim_true)
    plot_matrix_heatmap(
        np.array(A_enc),
        title=name,
        row_labels=[f"o{i}" for i in range(obs_dim_true)],
        col_labels=[f"s{i}" for i in range(num_states_true)],
        cmap="YlOrRd",
        ax=axes[idx],
    )

plt.suptitle("Encoder Representations Across Architectures", fontsize=13, y=1.05)
plt.tight_layout()
plt.show()

print("Look for:")
print("- Clear block structure = good encoding (different obs -> different states)")
print("- Uniform rows = collapse (all obs -> same state)")
print("- Noisy patterns = underfitting or training instability")

## Summary

In this notebook you have:

1. **Demonstrated the scaling problem** of tabular A/B matrices for high-dimensional observations
2. **Initialized and inspected** a `DeepGenerativeModel` with encoder and transition MLPs
3. **Trained the deep model** on synthetic data using `learn_deep_model()`
4. **Compared** deep vs tabular learning curves and final NLLs
5. **Extracted** approximate A matrices from the encoder to inspect learned representations
6. **Explored encoder collapse**: when and why the encoder ignores its input
7. **Compared encoder vs decoder** architectures conceptually
8. **Experimented** with different hidden dimensions and their effects on learning

### Key Takeaways

- **Tabular models** are optimal for small discrete problems but don't scale.
- **Deep models** trade sample efficiency for scalability: they work with
  continuous, high-dimensional observations but need more data.
- **Encoder collapse** is a real failure mode. Strategies to avoid it include:
  balanced encoder/transition capacity, lower learning rates, and decoder architectures.
- The entire deep model is **differentiable via JAX**, enabling end-to-end
  learning of the generative model from raw observation data.

### What's Next

With Modules 9-12 complete, you now have the full toolkit for Active Inference
in PGMax:
- Hand-specified models (Module 9)
- Understanding EFE decomposition (Module 10)
- Learning tabular models from data (Module 11)
- Scaling to high dimensions with deep models (Module 12)

The remaining modules in the curriculum will connect these tools to
multi-agent systems, hierarchical models, and real-world applications.